%% [markdown]
# 03e — NOMIS BRES Employment Audit
MSOA-level employment (LSOA level suppressed by ONS).
Distribution strategy: aggregate BRES to LA level, then distribute to LSOAs equally within LA.
Output: data/audit/lsoa_employment_proxy.parquet

In [1]:
# %%
import pandas as pd, numpy as np, re
from pathlib import Path
from difflib import get_close_matches

In [2]:
ROOT = Path("/Users/souravamseekarmarti/Projects/aequitas")
DATA = ROOT / "data"

In [3]:
# %%
bres = pd.read_csv(DATA / "raw/nomis/bres_msoa_2023.csv")
bres_eng = bres[bres['GEOGRAPHY_CODE'].str.startswith('E02')].copy()
print(f"England MSOAs: {len(bres_eng)}")
assert len(bres_eng) >= 6700, f"FAIL: expected ~6791, got {len(bres_eng)}"
assert bres_eng['OBS_VALUE'].isnull().sum() == 0, "FAIL: null employment values"
print(f"Total employees: {bres_eng['OBS_VALUE'].sum():,.0f}")

England MSOAs: 6791
Total employees: 27,343,200


In [4]:
# %%
# Extract LA name from MSOA name (remove trailing " NNN")
bres_eng['la_name_approx'] = bres_eng['GEOGRAPHY_NAME'].str.replace(r'\s+\d+$', '', regex=True)

In [5]:
# Aggregate to LA level
bres_la = bres_eng.groupby('la_name_approx', as_index=False)['OBS_VALUE'].sum()
bres_la.columns = ['la_name_bres', 'employment_total']
print(f"Distinct BRES LA names: {len(bres_la)}")
print(bres_la.head(5))

Distinct BRES LA names: 326
   la_name_bres  employment_total
0          Adur             21500
1     Allerdale             34200
2  Amber Valley             47300
3          Arun             49900
4      Ashfield             56300


In [6]:
# %%
master = pd.read_parquet(DATA / "audit/master_lsoa_table.parquet")
master_la = master[['lad_cd', 'lad_nm']].drop_duplicates('lad_cd')
print(f"Master LAs: {len(master_la)}")
print(master_la.head(3))

Master LAs: 296
        lad_cd                lad_nm
0    E09000001        City of London
4    E09000002  Barking and Dagenham
106  E09000003                Barnet


In [7]:
# %%
# Fuzzy match BRES LA names to master LAD names
master_names = master_la['lad_nm'].tolist()
name_map = {}
unmatched = []
for bn in bres_la['la_name_bres'].tolist():
    m = get_close_matches(bn, master_names, n=1, cutoff=0.8)
    if m:
        name_map[bn] = m[0]
    else:
        unmatched.append(bn)

In [8]:
print(f"Matched: {len(name_map)} / {len(bres_la)} LA names")
if unmatched:
    print(f"Unmatched ({len(unmatched)}): {unmatched[:15]}")

Matched: 290 / 326 LA names
Unmatched (36): ['Allerdale', 'Aylesbury Vale', 'Barrow-in-Furness', 'Bournemouth', 'Bristol', 'Carlisle', 'Chiltern', 'Christchurch', 'Copeland', 'Corby', 'Craven', 'Daventry', 'East Dorset', 'Eden', 'Forest Heath']


In [9]:
bres_la['lad_nm'] = bres_la['la_name_bres'].map(name_map)

In [10]:
# De-duplicate: if multiple BRES LA names map to same lad_nm, sum them
bres_la_dedup = bres_la.dropna(subset=['lad_nm']).groupby('lad_nm', as_index=False)['employment_total'].sum()
print(f"Deduplicated BRES LA matches: {len(bres_la_dedup)}")

Deduplicated BRES LA matches: 284


In [11]:
# %%
# Merge employment totals to master on lad_nm, then distribute equally within each LA
master_emp = master[['lsoa_cd', 'lsoa_nm', 'lad_cd', 'lad_nm']].copy()
master_emp = master_emp.merge(bres_la_dedup[['lad_nm', 'employment_total']], on='lad_nm', how='left')

In [12]:
# Count LSOAs per LA for equal distribution
lsoa_counts = master_emp.groupby('lad_nm')['lsoa_cd'].count().rename('lsoa_count')
master_emp = master_emp.join(lsoa_counts, on='lad_nm')
master_emp['employment_proxy'] = master_emp['employment_total'] / master_emp['lsoa_count']

In [13]:
coverage = master_emp['employment_proxy'].notna().sum()
coverage_rate = coverage / len(master_emp) * 100
print(f"LSOAs with employment proxy: {coverage} / {len(master_emp)} ({coverage_rate:.1f}%)")
assert coverage_rate >= 85.0, f"FAIL: coverage {coverage_rate:.1f}% below 85%"
print("CHECK PASS: coverage >= 85%")

LSOAs with employment proxy: 31217 / 33755 (92.5%)
CHECK PASS: coverage >= 85%


In [14]:
# %%
out = master_emp[['lsoa_cd', 'lsoa_nm', 'lad_cd', 'lad_nm', 'employment_total', 'employment_proxy']].copy()
out.to_parquet(DATA / "audit/lsoa_employment_proxy.parquet", index=False)
print(f"Saved lsoa_employment_proxy.parquet: {len(out)} LSOAs")
print(out['employment_proxy'].describe())

Saved lsoa_employment_proxy.parquet: 33755 LSOAs
count     31217.000000
mean        815.496364
std        1589.682808
min         146.638655
25%         590.522876
50%         694.927536
75%         868.750000
max      110166.666667
Name: employment_proxy, dtype: float64


In [15]:
# %%
print("=== BRES Employment Audit Summary ===")
print(f"England MSOAs:           {len(bres_eng)}")
print(f"Total employees (BRES):  {bres_eng['OBS_VALUE'].sum():,.0f}")
print(f"LAs matched:             {len(bres_la_dedup)}")
print(f"LSOAs with proxy:        {coverage} ({coverage_rate:.1f}%)")
print("NOTE: Proxy = LA total employment / number of LSOAs in that LA (equal distribution)")
print("NOTE: LSOA-level BRES suppressed by ONS — this is best available approximation")
print("03e_employment_bres COMPLETE")

=== BRES Employment Audit Summary ===
England MSOAs:           6791
Total employees (BRES):  27,343,200
LAs matched:             284
LSOAs with proxy:        31217 (92.5%)
NOTE: Proxy = LA total employment / number of LSOAs in that LA (equal distribution)
NOTE: LSOA-level BRES suppressed by ONS — this is best available approximation
03e_employment_bres COMPLETE
